# NB1｜第一條光譜：認識工具與資料

**食品分析｜拉曼光譜與 RamanSPy 入門系列（第 1 本，共 5 本）**

拉曼光譜像是分子的「指紋」。這一本先讓你把電腦環境準備好，並成功畫出人生第一條拉曼光譜。

---
### 這一本你會學到
- 在 Google Colab 執行 Python（不用安裝任何東西）
- 知道拉曼光譜的 X 軸、Y 軸各代表什麼
- 用 `ramanspy` 讀進一條光譜並畫出來
- 看懂「原始光譜」上的三個問題

> 💡 **完全沒寫過程式也沒關係。** 你只要做三件事：
> 1. 用滑鼠點每一格左邊的 ▶ 播放鍵（或按 `Shift + Enter`）
> 2. 看下面跑出來的圖和數字
> 3. 遇到 `# 👉 換你做` 的地方，照提示改一個數字或一個字，再跑一次


## 0. 暖身：Python 到底在做什麼？

把 Python 想成一台很聽話但很笨的實驗助理：

| 你在實驗室說 | 在 Python 寫成 |
|---|---|
| 「把這個叫做 A」 | `A = 3` |
| 「把這張表讀進來」 | `pd.read_csv("檔名.csv")` |
| 「畫成圖給我看」 | `rp.plot.spectra(...)` |
| 「這是註解，你不用理」 | `# 前面加井字號` |

只要照著跑，看得懂結果，就達到這門課的目標了。

## 1. 環境準備

點下面這一格左邊的 ▶ 執行。第一次會跑約 1 分鐘。

In [ ]:
# ===== 第一次執行請先跑這一格（大約 1 分鐘）=====
# 在 Google Colab 上，套件不是永久安裝的，每次重開都要跑一次。
!pip install -q ramanspy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ramanspy as rp

# 讓圖上的中文正常顯示（Colab 用）
!wget -q -O TaipeiSans.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_ 2>/dev/null
import matplotlib
try:
    matplotlib.font_manager.fontManager.addfont("TaipeiSans.ttf")
    matplotlib.rc("font", family="Taipei Sans TC Beta")
except Exception:
    pass
matplotlib.rcParams["axes.unicode_minus"] = False

print("準備完成！")


## 2. 拉曼光譜的兩個軸

- **X 軸：拉曼位移（Raman shift, cm⁻¹）** — 對應「哪一種化學鍵在振動」。這是分子的身分證號碼，跟雷射波長無關。
- **Y 軸：強度（Intensity, a.u.）** — 大致對應「這種鍵有多少」。但**單位是任意單位**，不同天、不同儀器不能直接比大小，所以後面一定要做歸一化。

食品裡最常用到的區段是 **400–1800 cm⁻¹**，叫做**指紋區（fingerprint region）**。

## 3. 讀入一條真實感的奶粉光譜

In [ ]:
# ===== 資料載入設定 =====
# 這一行由老師部署時自動填入正確的 GitHub 網址，學生不用改。
DATA_BASE = "https://raw.githubusercontent.com/Tai-ShengYeh/Tai-ShengYeh.github.io/main/ramanspy-food-analysis/data/"

# 若你把 CSV 直接上傳到 Colab 左側「檔案」，把上面那行改成： DATA_BASE = ""
# 若你在自己電腦跑，且 data 資料夾就在旁邊，改成：       DATA_BASE = "data/"

def load_spectra(filename):
    """讀 CSV → 回傳 (樣品資訊表 meta, ramanspy 光譜物件 spectra)"""
    df = pd.read_csv(DATA_BASE + filename)
    meta_cols = [c for c in df.columns if not c.replace(".", "", 1).isdigit()]
    axis = np.array([float(c) for c in df.columns if c not in meta_cols])
    spectra = rp.SpectralContainer(df.drop(columns=meta_cols).values, axis)
    return df[meta_cols].reset_index(drop=True), spectra

print("load_spectra() 已定義，資料來源：", DATA_BASE or "（Colab 本機檔案）")


In [ ]:
raw = pd.read_csv(DATA_BASE + "milk_powder_raw_single.csv")
raw.head()      # 看前 5 列：第一欄是拉曼位移，第二欄是強度

In [ ]:
print("總共有", len(raw), "個資料點")
print("拉曼位移範圍：", raw.iloc[:, 0].min(), "~", raw.iloc[:, 0].max(), "cm-1")

### 建立 ramanspy 的光譜物件

`rp.Spectrum(強度, 拉曼位移)` 會把兩欄數字包成一個「光譜物件」，之後所有分析都吃這個物件。

⚠️ **順序不要寫反**：先強度、後 X 軸。

In [ ]:
x = raw["raman_shift_cm-1"].values     # X 軸
y = raw["intensity"].values           # Y 軸

spectrum = rp.Spectrum(y, x)

rp.plot.spectra(spectrum, title="奶粉（原始光譜）")
rp.plot.show()

## 4. 看圖說故事：這張圖有三個問題

仔細看剛剛那張圖：

| 問題 | 長什麼樣 | 為什麼會有 | 之後怎麼解決 |
|---|---|---|---|
| **螢光背景** | 整條線像坐在一個大駝峰上 | 食品裡的色素、蛋白質受雷射激發放螢光，強度是拉曼訊號的數百倍 | 基線校正（baseline correction）|
| **宇宙射線尖峰** | 一兩根又高又細的針 | 高能粒子直接打到 CCD 偵測器 | 去尖峰（despiking）|
| **隨機雜訊** | 線條毛毛的 | 偵測器本身的電子雜訊 | 平滑（smoothing）|

只有把這三個拿掉，剩下的凸起才是**真正的化學資訊**。這正是 NB2 要做的事。

## 5. 👉 換你做

把下面 `START` 和 `END` 改成不同數字，看看不同區段。
試著找出：**哪一個區段的峰最多？**

In [ ]:
START = 800     # 👉 換你做：改改看，例如 400 / 1000 / 1400
END   = 1200    # 👉 換你做：改改看，例如 700 / 1300 / 1800

plt.figure(figsize=(7, 3))
mask = (x >= START) & (x <= END)
plt.plot(x[mask], y[mask])
plt.xlabel("拉曼位移 (cm$^{-1}$)")
plt.ylabel("強度 (a.u.)")
plt.title(f"{START}–{END} cm$^{{-1}}$")
plt.show()

### 🧪 自我檢核（做完再往下）

1. 拉曼位移的單位是什麼？它跟雷射波長有沒有關係？
2. 為什麼 Y 軸的強度不能直接拿來比較兩天測的樣品？
3. 原始光譜上那根又高又細的針，是樣品裡的化學成分嗎？

<details><summary>▶ 點開看參考答案</summary>

1. cm⁻¹（波數）。拉曼位移是「入射光與散射光的能量差」，**與雷射波長無關**，所以同一物質用不同雷射測，峰位一樣。
2. 因為 Y 軸是任意單位（a.u.），受雷射功率、曝光時間、聚焦深度影響。要比較必須先歸一化。
3. 不是。那是宇宙射線打到偵測器造成的假訊號，特徵是**極窄（只有 1–2 個資料點）**，真正的拉曼峰至少有數個點寬。

</details>


---
### 📚 這一本用到的資料與文獻

**資料**：`data/` 內的光譜為**依文獻峰位建立的模擬資料**（`make_data.py`，亂數種子 20260801），刻意加入螢光背景、宇宙射線與雜訊。可用於教學演練，**不可引用為實驗證據**。

**主要文獻**

- Georgiev, D. et al. *RamanSPy: An Open-Source Python Package for Integrative Raman Spectroscopy Data Analysis*. **Anal. Chem.** 2024, 96(21), 8492–8500. doi:10.1021/acs.analchem.4c00383
- Gill, D.; Kilponen, R. G.; Rimai, L. *Resonance Raman Scattering … in Intact Plant Tissues*. **Nature** 1970, 227, 743–744. doi:10.1038/227743a0
- Lu, L. et al. *Resonance Raman scattering of β-carotene … second singlet state*. **J. Photochem. Photobiol. B** 2018, 179, 18–22. doi:10.1016/j.jphotobiol.2017.12.022
- Withnall, R. et al. *Raman spectra of carotenoids in natural products*. **Spectrochim. Acta A** 2003, 59(10), 2207–2212. doi:10.1016/S1386-1425(03)00064-7
- de Oliveira, V. E. et al. *Carotenes and carotenoids in natural biological samples*. **J. Raman Spectrosc.** 2010, 41(6), 642–650. doi:10.1002/jrs.2493
- Portarena, S. et al. *Cultivar discrimination, fatty acid profile and carotenoid characterization of monovarietal olive oils by Raman spectroscopy at a single glance*. **Food Control** 2019, 96, 137–145. doi:10.1016/j.foodcont.2018.09.011
- Chen, Y. et al. *Quantitative analysis of β-carotene and unsaturated fatty acids in blended olive oil via Raman spectroscopy combined with model prediction*. **Food Chemistry** 2025, 470, 142621. doi:10.1016/j.foodchem.2024.142621
- Schmidt, W. et al. *Continuous Temperature-Dependent Raman Spectroscopy of Melamine and Structural Analog Detection in Milk Powder*. **Appl. Spectrosc.** 2015, 69(3), 398–406. doi:10.1366/14-07600
- Zhang, X. et al. *Detection of melamine in liquid milk using SERS*. **J. Raman Spectrosc.** 2010, 41(12), 1655–1660. doi:10.1002/jrs.2629
- Kim, A. et al. *Melamine Sensing in Milk Products by Using SERS*. **Anal. Chem.** 2012, 84(21), 9303–9309. doi:10.1021/ac302025q
- FAO/WHO Codex Alimentarius. *General Standard for Contaminants and Toxins in Food and Feed*, **CXS 193-1995**.

完整清單見課程網站的「數據來源」與「參考文獻」兩節。